# Experimental test 6

* test difference btw Self_Consistency and Self_Consistency_re(refactored ver)
* for Self_Consistency_re 
    test ('vl',              # llm_model
        3,                # few_shot_n
        5,                # test_n(# of question for test)
        'Y',              # q_src_yn 
        5,                # iteration num
        'sys_prompt10',   # prompt ver
        3,                # self-consistency number
        0.01,             # temperature
        'ver7'            # excel_verion
        )
* for Self_Consistency
        test ('vl',              # llm_model
            3,                # few_shot_n
            5,                # test_n(# of question for test)
            'Y',              # q_src_yn 
            6,                # iteration num
            'sys_prompt10',   # prompt ver
            3,                # self-consistency number
            0.01,             # temperature
            'ver7'            # excel_verion
            )
* compare the accuracy score 


In [1]:
import os
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [ ]:
def sc_calc_acc_condition_with_temp_with_sc(llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')]

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [3]:
    # test ('vl',              # llm_model
    #     3,                # few_shot_n
    #     5,                # test_n(# of question for test)
    #     'Y',              # q_src_yn 
    #     5,                # iteration num
    #     'sys_prompt10',   # prompt ver
    #     3,                # self-consistency number
    #     0.01,             # temperature
    #     'ver7'            # excel_verion
    #     )

In [4]:
# sc_vl_result_3_60_Y_100_sys_prompt10_5_0.01_ver7_99.csv
# stop        = ["</Difficulty Level>"] 있는 버전의 점수 
list_, df_ =         sc_calc_acc_condition_with_temp_with_sc('vl', 3, 5, 'Y', 5, 'sys_prompt10', 3,  0.01, 'ver7')
print(list_)

size of the dataset : 3
size of the dataset : 4
size of the dataset : 3
size of the dataset : 3
size of the dataset : 4
              precision    recall  f1-score   support

           0      1.000     0.875     0.933         8
           1      0.857     0.857     0.857         7
           2      0.667     1.000     0.800         2

    accuracy                          0.882        17
   macro avg      0.841     0.911     0.863        17
weighted avg      0.902     0.882     0.886        17

vl_result_3_5_Y :  88.23529411764706
[np.float64(100.0), np.float64(100.0), np.float64(100.0), np.float64(66.66666666666666), np.float64(75.0)]


In [6]:
        # test ('vl',              # llm_model
        #     3,                # few_shot_n
        #     5,                # test_n(# of question for test)
        #     'Y',              # q_src_yn 
        #     6,                # iteration num
        #     'sys_prompt10',   # prompt ver
        #     3,                # self-consistency number
        #     0.01,             # temperature
        #     'ver7'            # excel_verion
        #     )

In [8]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('vl', 3, 5, 'Y', 6, 'sys_prompt10', 3,  0.01, 'ver7')

size of the dataset : 3
size of the dataset : 2
size of the dataset : 4
size of the dataset : 3
size of the dataset : 3
size of the dataset : 2
              precision    recall  f1-score   support

           0      1.000     0.818     0.900        11
           1      0.667     0.800     0.727         5
           2      0.500     1.000     0.667         1

    accuracy                          0.824        17
   macro avg      0.722     0.873     0.765        17
weighted avg      0.873     0.824     0.835        17

vl_result_3_5_Y :  82.35294117647058


In [ ]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('vl', 3, 100, 'Y', 50, 'sys_prompt10', 5,  0.01, 'ver7')
# print(list_)

size of the dataset : 47
size of the dataset : 52
size of the dataset : 54
size of the dataset : 55
              precision    recall  f1-score   support

           0      0.980     0.807     0.885       119
           1      0.701     0.857     0.771        63
           2      0.788     1.000     0.881        26

    accuracy                          0.846       208
   macro avg      0.823     0.888     0.846       208
weighted avg      0.871     0.846     0.850       208

vl_result_3_100_Y :  84.61538461538461
([np.float64(85.1063829787234), np.float64(86.53846153846155), np.float64(83.33333333333334), np.float64(83.63636363636363)],            id                                           question  \
0    71389500  <Title>Kubernetes: Error loading ASGI app. Att...   
1    71389500  <Title>Kubernetes: Error loading ASGI app. Att...   
2    71389500  <Title>Kubernetes: Error loading ASGI app. Att...   
3    71389500  <Title>Kubernetes: Error loading ASGI app. Att...   
4    71389500 